In [1]:
from datasets import DatasetDict, Dataset
from utils.dataset_utils import load_or_download_dataset, save2Local
import os

/home/velgor/Documents/Code/AI8vo/parser-python/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = load_or_download_dataset("esquivelgor/F1_SBL_T_60", split="test")
dataset

📂 Loading dataset from local file: ./datasets/esquivelgor/F1_SBL_T_60.json


Dataset({
    features: ['repo', 'instance_id', 'base_commit', 'patch', 'test_patch', 'problem_statement', 'hints_text', 'created_at', 'version', 'FAIL_TO_PASS', 'PASS_TO_PASS', 'environment_setup_commit'],
    num_rows: 300
})

---
### F1 - Filter by number of affected files 

In this step, we are going to filter all the examples to get the ones that only affect one file.

In [21]:
import re
import requests
from IPython.display import clear_output
import logging

# Basic logging config
logging.basicConfig(level=logging.DEBUG, format='%(message)s')
logging.getLogger("urllib3").setLevel(logging.WARNING)
logging.getLogger("requests").setLevel(logging.WARNING)

def extract_file_line_ranges(patch_text):
    results = []
    current_file = None

    for line in patch_text.splitlines():
        if line.startswith('+++ b/'):
            current_file = line[6:]
        elif line.startswith('@@') and current_file:
            match = re.search(r'\+(\d+)(?:,(\d+))?', line)
            if match:
                start_line = int(match.group(1))
                line_count = int(match.group(2)) if match.group(2) else 1
                end_line = start_line + line_count - 1
                results.append([current_file, start_line, end_line])
    return results

def filter_single_file_patches(dataset):
    filtered_data = []

    successfulRes = 0
    errorRes = 0
    total = 0 

    for example in dataset:
        ranges = extract_file_line_ranges(example['patch'])
        files = set(r[0] for r in ranges)
        
        if len(files) == 1:
            file = next(iter(files))
            url = f"https://raw.githubusercontent.com/{example['repo']}/{example['base_commit']}/{file}"

            response = requests.get(url)

            if response.status_code == 200:
                successfulRes += 1
                line_ranges = [[r[1], r[2]] for r in ranges if r[0] == file]
                example["target_file"] = response.content.decode('utf-8')
                example["patch_range"] = line_ranges
                example["file_path"] = file

                filtered_data.append(example)
            else:
                errorRes += 1
        else:
            errorRes += 1
        
        clear_output(wait=True)
        total += 1
        print(f"\n🐛 Dataset size: {len(dataset)}")
        print(f"🔍 Processing row: {total}/{len(dataset)}")
        print(f"🔍 Evaluation: {(total / len(dataset)) * 100:.2f}%")
        print(f"✅ Success rate: {(successfulRes / total) * 100:.2f}%")


    logging.info("✅ Task completed successfully!")
    logging.info(f"✅ Length of filtered dataset: {len(filtered_data)} rows.")
    return Dataset.from_list(filtered_data)

In [23]:
F1Dataset = filter_single_file_patches(dataset)

✅ Task completed successfully!
✅ Length of filtered dataset: 300 rows.



🐛 Dataset size: 300
🔍 Processing row: 300/300
🔍 Evaluation: 100.00%
✅ Success rate: 100.00%


In [ ]:
n = str(len(F1Dataset))
save2Local(F1Dataset, "./datasets", "esquivelgor/F1_SBL", n, split="TE")

open file: /home/velgor/Documents/Code/AI8vo/parser-python/datasets/esquivelgor/F1_SBL_TE_300.json
Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00,  6.16ba/s]

✅ Dataset saved to ./datasets/esquivelgor/F1_SBL_TE_300.json


---
### F2 - FIlter by OOP python files

In [25]:
from OOPFilter import OOP_filter

datasetF2 = []

def checkOOP(datasetFiltered):
    OOP = 0
    NOOP = 0
    failedExamples = 0 
    
    for row in datasetFiltered:
        if 'target_file' in row:
            isOOP = OOP_filter(row['target_file'])
            row['OOP'] = isOOP
            if isOOP == 1:
                datasetF2.append(row) 
                OOP += 1
            elif isOOP == -1:
                failedExamples += 1
            else:
                NOOP += 1
        else:
            row['OOP'] = None

    print(f"Is OOP: {OOP}\nNot OOP: {NOOP}\nFailed: {failedExamples}\n")
    print(f"✅ Success rate: {OOP / len(datasetFiltered) * 100:.2f}%")
    return Dataset.from_list(datasetF2)

In [26]:
F1Dataset

Dataset({
    features: ['repo', 'instance_id', 'base_commit', 'patch', 'test_patch', 'problem_statement', 'hints_text', 'created_at', 'version', 'FAIL_TO_PASS', 'PASS_TO_PASS', 'environment_setup_commit', 'target_file', 'patch_range', 'file_path'],
    num_rows: 300
})

In [27]:
#dataset = load_or_download_dataset("esquivelgor/F1_SB_TR_1691", split="train")

datasetF2 = checkOOP(F1Dataset)
n = str(len(datasetF2))
save2Local(datasetF2, "./datasets", "esquivelgor/F2_SBL", n, split="TE")

datasetF2

open file: /home/velgor/Documents/Code/AI8vo/parser-python/datasets/esquivelgor/F2_SBL_TE_84.json


Is OOP: 84
Not OOP: 24
Failed: 192

✅ Success rate: 28.00%


Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 38.69ba/s]

✅ Dataset saved to ./datasets/esquivelgor/F2_SBL_TE_84.json


Dataset({
    features: ['repo', 'instance_id', 'base_commit', 'patch', 'test_patch', 'problem_statement', 'hints_text', 'created_at', 'version', 'FAIL_TO_PASS', 'PASS_TO_PASS', 'environment_setup_commit', 'target_file', 'patch_range', 'file_path', 'OOP'],
    num_rows: 84
})

Upload dataset to Huggingface

In [30]:
datasetPath = "esquivelgor/F2_SBL_TE_84"
split = "test"

dataset = load_or_download_dataset(datasetPath, split=split)

dataset = dataset.map(lambda x: {**x, "image_name": "python:3.11"})
dataset.push_to_hub(datasetPath, split=split, private=False)

📂 Loading dataset from local file: ./datasets/esquivelgor/F2_SBL_TE_84.json


Attempting to acquire lock 136766147109392 on /home/velgor/.cache/huggingface/datasets/_home_velgor_.cache_huggingface_datasets_json_default-c312310f252ddfda_0.0.0_f4e89e8750d5d5ffbef2c078bf0ddfedef29dc2faff52a6255cf513c05eb1092.lock
Lock 136766147109392 acquired on /home/velgor/.cache/huggingface/datasets/_home_velgor_.cache_huggingface_datasets_json_default-c312310f252ddfda_0.0.0_f4e89e8750d5d5ffbef2c078bf0ddfedef29dc2faff52a6255cf513c05eb1092.lock
Attempting to release lock 136766147109392 on /home/velgor/.cache/huggingface/datasets/_home_velgor_.cache_huggingface_datasets_json_default-c312310f252ddfda_0.0.0_f4e89e8750d5d5ffbef2c078bf0ddfedef29dc2faff52a6255cf513c05eb1092.lock
Lock 136766147109392 released on /home/velgor/.cache/huggingface/datasets/_home_velgor_.cache_huggingface_datasets_json_default-c312310f252ddfda_0.0.0_f4e89e8750d5d5ffbef2c078bf0ddfedef29dc2faff52a6255cf513c05eb1092.lock
Attempting to acquire lock 136766148623584 on /home/velgor/.cache/huggingface/datasets/jso

CommitInfo(commit_url='https://huggingface.co/datasets/esquivelgor/F2_SBL_TE_84/commit/655b06065b75e2230bf4ea845a9115fe5b414d61', commit_message='Upload dataset', commit_description='', oid='655b06065b75e2230bf4ea845a9115fe5b414d61', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/esquivelgor/F2_SBL_TE_84', endpoint='https://huggingface.co', repo_type='dataset', repo_id='esquivelgor/F2_SBL_TE_84'), pr_revision=None, pr_num=None)

In [29]:
datasetF2[0]


{'repo': 'astropy/astropy',
 'instance_id': 'astropy__astropy-14182',
 'base_commit': 'a5917978be39d13cd90b517e1de4e7a539ffaa48',
 'patch': 'diff --git a/astropy/io/ascii/rst.py b/astropy/io/ascii/rst.py\n--- a/astropy/io/ascii/rst.py\n+++ b/astropy/io/ascii/rst.py\n@@ -27,7 +27,6 @@ def get_fixedwidth_params(self, line):\n \n \n class SimpleRSTData(FixedWidthData):\n-    start_line = 3\n     end_line = -1\n     splitter_class = FixedWidthTwoLineDataSplitter\n \n@@ -39,12 +38,29 @@ class RST(FixedWidth):\n \n     Example::\n \n-        ==== ===== ======\n-        Col1  Col2  Col3\n-        ==== ===== ======\n-          1    2.3  Hello\n-          2    4.5  Worlds\n-        ==== ===== ======\n+      >>> from astropy.table import QTable\n+      >>> import astropy.units as u\n+      >>> import sys\n+      >>> tbl = QTable({"wave": [350, 950] * u.nm, "response": [0.7, 1.2] * u.count})\n+      >>> tbl.write(sys.stdout,  format="ascii.rst")\n+      ===== ========\n+       wave response\n+   